# Finding and Fixing Pipeline Inefficiency with Cortex Code

This notebook builds a **deliberately inefficient data pipeline**, then hands it to
Cortex Code to investigate, diagnose and fix.

The pipeline reproduces four patterns that appear constantly in production
ingestion workloads:

| Pattern | What it looks like here |
|---|---|
| A redundant full-table pass inside an incremental MERGE | 12.6 GB read to process a 5,000-row batch |
| No partition pruning on the join key | 100% of micro-partitions scanned, every run |
| Fixed-cadence jobs with no change predicate | Tasks fire every 1-3 minutes whether or not data arrived |
| A warehouse resize used as the fix | Runtime improved, scan volume unchanged, cost doubled |
| A stored-procedure task failing with no retry policy | ~35% failure rate, invisible unless you look |

Everything is synthetic, generated in-account. No external data.

## Story at a glance

```text
                         fixed-cadence tasks
                    +---------------------------+
                    | every 1-5 minutes          |
                    +-------------+-------------+
                                  |
        +-------------------------+-------------------------+
        |                         |                         |
        v                         v                         v
  +-----------+             +-----------+             +-------------+
  | Feed      |             | Feed      |             | Reporting   |
  | pipeline  |             | pipeline  |             | refresh     |
  | X-Small   |             | Small     |             | X-Small     |
  +-----+-----+             +-----+-----+             +------+------+
        |                         |                         |
        +------------+------------+-------------------------+
                     v
             +-------------------+
             | Staging batches   |  5,000 rows/run
             +---------+---------+
                       v
             +-------------------+
             | MERGE procedure   |  full-table window pass
             +---------+---------+
                       v
             +-------------------+
             | Transaction ledger|  20M rows / wide payload
             +---------+---------+
                       |
             +---------v---------+
             | Scan + spill      |  cost and latency signal
             +-------------------+
```

The demo follows this path: **schedule -> stage -> MERGE -> ledger -> telemetry**.

---

## How to use this notebook

1. Run **Part 1** to build the environment (~10 minutes).
2. Run **Part 2** to start the pipeline and let it soak for 60 minutes.
3. Run **Part 3** to confirm the problem is measurable.
4. Work through the prompts in **Part 4** with Cortex Code.
5. Run **Part 5** to prove the fix, **Part 6** to stop and clean up.

## Cost and safety

| | |
|---|---|
| One-off build | ~2 credits |
| While the pipeline runs | ~3.5 credits/hour |
| Idle (built, tasks suspended) | 0 credits |
| Hard ceiling | 40 credits/day, enforced by a resource monitor |

Three safety properties are built in:

- A resource monitor suspends every demo warehouse at its quota.
- Tasks are created **suspended** — nothing recurring runs until you choose to start it.
- **Part 6 is a one-click stop.** Run it the moment the session ends.

> Requires `ACCOUNTADMIN` (or a role with `CREATE WAREHOUSE`, `CREATE DATABASE`
> and `CREATE RESOURCE MONITOR`). Run in a demo or sandbox account, never in production.

---
# Part 1 — Build the environment

## 1.1 Cost guard

One resource monitor covering every demo warehouse, with a hard
`SUSPEND_IMMEDIATE` at 100%. This is what makes the rest of the notebook safe
to leave running.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['CREATE OR REPLACE RESOURCE MONITOR DEMO_COST_GUARD\n  WITH CREDIT_QUOTA = 40\n       FREQUENCY = DAILY\n       START_TIMESTAMP = IMMEDIATELY\n  TRIGGERS ON 50  PERCENT DO NOTIFY\n           ON 75  PERCENT DO NOTIFY\n           ON 90  PERCENT DO NOTIFY\n           ON 100 PERCENT DO SUSPEND_IMMEDIATE']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.2 Warehouses

Two ingestion tiers running **identical work at different sizes**. That pairing
is the point: it lets us test later whether the resize actually addressed
anything.

`AUTO_SUSPEND = 300` against a 1-3 minute task cadence means these warehouses
never suspend while the pipeline runs — reproducing the "warehouse resident
around the clock, yet still spilling" signature. It also means they drain within
five minutes once the tasks stop.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE WAREHOUSE DEMO_INGEST_WH_XS\n    WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300 AUTO_RESUME = TRUE\n    INITIALLY_SUSPENDED = TRUE STATEMENT_TIMEOUT_IN_SECONDS = 900\n    RESOURCE_MONITOR = DEMO_COST_GUARD\n    COMMENT = 'DEMO - synthetic ingestion tier (X-Small)'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE WAREHOUSE DEMO_INGEST_WH_S\n    WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 300 AUTO_RESUME = TRUE\n    INITIALLY_SUSPENDED = TRUE STATEMENT_TIMEOUT_IN_SECONDS = 900\n    RESOURCE_MONITOR = DEMO_COST_GUARD\n    COMMENT = 'DEMO - synthetic ingestion tier (Small, the resized one)'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE WAREHOUSE DEMO_REPORTING_WH\n    WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300 AUTO_RESUME = TRUE\n    INITIALLY_SUSPENDED = TRUE STATEMENT_TIMEOUT_IN_SECONDS = 900\n    RESOURCE_MONITOR = DEMO_COST_GUARD\n    COMMENT = 'DEMO - synthetic reporting tier'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


A temporary `LARGE` warehouse for the one-off data build, so the generation
work does not pollute the ingestion tiers' telemetry. Dropped in step 1.9.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE WAREHOUSE DEMO_BUILD_WH\n    WAREHOUSE_SIZE = 'LARGE' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE\n    INITIALLY_SUSPENDED = TRUE STATEMENT_TIMEOUT_IN_SECONDS = 1800\n    RESOURCE_MONITOR = DEMO_COST_GUARD\n    COMMENT = 'DEMO - temporary build warehouse, dropped after setup'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.3 Database and schema

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE DATABASE PIPELINE_COST_DEMO\n    COMMENT = 'DEMO - synthetic inefficient pipeline for Cortex Code demonstration'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['CREATE OR REPLACE SCHEMA PIPELINE_COST_DEMO.INGESTION']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['USE DATABASE PIPELINE_COST_DEMO',
    'USE SCHEMA INGESTION']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.4 The target table

This is where the inefficiency is designed in, and it is worth being explicit
about how — because all three choices are things real pipelines do by accident.

1. **`TXN_KEY` is MD5-derived**, so key values scatter uniformly across every
   micro-partition. Each partition's min/max on `TXN_KEY` spans almost the whole
   key domain, which makes partition pruning on that column impossible.
2. **No cluster key on the join column.**
3. **A wide, incompressible `PAYLOAD` column**, which inflates bytes read per
   partition and is what eventually drives the query to spill to disk.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TABLE TRANSACTION_LEDGER (\n    TXN_KEY        STRING,          -- MERGE join key, deliberately unclustered\n    ENTITY_ID      STRING,\n    PRODUCT_ID     STRING,\n    REGION         STRING,\n    QUANTITY       NUMBER(18,4),\n    AMOUNT         NUMBER(18,2),\n    TXN_STATUS     STRING,\n    VERSION_NO     NUMBER,\n    LAST_UPDATED   TIMESTAMP_NTZ,\n    PAYLOAD        STRING           -- wide incompressible column\n)\nCOMMENT = 'DEMO - synthetic transaction ledger. No cluster key on TXN_KEY (deliberate).'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.5 Generate the ledger

20M rows, about 6.3 GB across ~415 micro-partitions. **Takes roughly 4 minutes**
on the `LARGE` build warehouse.

Do not reduce the row count much below 20M. At 8M rows the sort in the bad query
fits in memory and the spill signature disappears entirely, which removes one of
the four findings.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['USE WAREHOUSE DEMO_BUILD_WH',
    "INSERT INTO TRANSACTION_LEDGER\nSELECT\n    'TXN-' || MD5(seq::STRING),\n    'ENT-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 7, '0'),\n    'PRD-' || LPAD(UNIFORM(1, 50000,  RANDOM())::STRING, 6, '0'),\n    ARRAY_CONSTRUCT('NA-EAST','NA-WEST','UK','DE','FR','JP','HK','AU','SG','BR')\n        [UNIFORM(0, 9, RANDOM())]::STRING,\n    UNIFORM(1, 500000, RANDOM()),\n    UNIFORM(1000, 99999999, RANDOM()) / 100,\n    ARRAY_CONSTRUCT('COMPLETE','PENDING','FAILED','PARTIAL')\n        [UNIFORM(0, 3, RANDOM())]::STRING,\n    1,\n    DATEADD(second, -UNIFORM(0, 7776000, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,\n    RANDSTR(400, RANDOM())\nFROM (SELECT SEQ8() AS seq FROM TABLE(GENERATOR(ROWCOUNT => 20000000)))"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.6 The incoming feed

A small bounded batch per run. This is precisely what makes the full-table
rescan so wasteful: tiny input, enormous read.

The key domain is 24M against a 20M-row ledger, giving roughly 83% updates and
17% inserts per batch.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TABLE TRANSACTION_FEED_STAGING (\n    PIPELINE_NAME STRING,\n    TXN_KEY STRING, ENTITY_ID STRING, PRODUCT_ID STRING, REGION STRING,\n    QUANTITY NUMBER(18,4), AMOUNT NUMBER(18,2), TXN_STATUS STRING,\n    LAST_UPDATED TIMESTAMP_NTZ, PAYLOAD STRING, BATCH_ID STRING\n) COMMENT = 'DEMO - synthetic incremental feed batches isolated by pipeline'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TABLE PIPELINE_RUN_LOG (\n    RUN_ID STRING, PIPELINE_NAME STRING, STARTED_AT TIMESTAMP_NTZ,\n    ENDED_AT TIMESTAMP_NTZ, ROWS_IN_BATCH NUMBER, STATUS STRING, MESSAGE STRING\n) COMMENT = 'DEMO - application-level run log for the synthetic pipeline'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE PROCEDURE SP_GENERATE_FEED_BATCH(P_BATCH_ROWS NUMBER, P_PIPELINE_NAME STRING)\nRETURNS STRING\nLANGUAGE SQL\nAS\n$$\nBEGIN\n    DELETE FROM TRANSACTION_FEED_STAGING\n    WHERE PIPELINE_NAME = :P_PIPELINE_NAME;\n\n    INSERT INTO TRANSACTION_FEED_STAGING\n    WITH RAW_BATCH AS (\n        SELECT\n            :P_PIPELINE_NAME                                             AS PIPELINE_NAME,\n            'TXN-' || MD5(UNIFORM(1, 24000000, RANDOM())::STRING)      AS TXN_KEY,\n            'ENT-' || LPAD(UNIFORM(1, 200000, RANDOM())::STRING, 7, '0') AS ENTITY_ID,\n            'PRD-' || LPAD(UNIFORM(1, 50000,  RANDOM())::STRING, 6, '0') AS PRODUCT_ID,\n            ARRAY_CONSTRUCT('NA-EAST','NA-WEST','UK','DE','FR','JP','HK','AU','SG','BR')\n                [UNIFORM(0, 9, RANDOM())]::STRING                       AS REGION,\n            UNIFORM(1, 500000, RANDOM())                                AS QUANTITY,\n            UNIFORM(1000, 99999999, RANDOM()) / 100                     AS AMOUNT,\n            ARRAY_CONSTRUCT('COMPLETE','PENDING','FAILED','PARTIAL')\n                [UNIFORM(0, 3, RANDOM())]::STRING                       AS TXN_STATUS,\n            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ                          AS LAST_UPDATED,\n            RANDSTR(400, RANDOM())                                      AS PAYLOAD,\n            UUID_STRING()                                               AS BATCH_ID,\n            SEQ8()                                                       AS SEQ\n        FROM TABLE(GENERATOR(ROWCOUNT => 5000))\n    )\n    SELECT PIPELINE_NAME, TXN_KEY, ENTITY_ID, PRODUCT_ID, REGION, QUANTITY,\n           AMOUNT, TXN_STATUS, LAST_UPDATED, PAYLOAD, BATCH_ID\n    FROM RAW_BATCH\n    QUALIFY ROW_NUMBER() OVER (PARTITION BY TXN_KEY ORDER BY SEQ) = 1;\n\n    RETURN 'feed batch generated for ' || :P_PIPELINE_NAME;\nEND;\n$$"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.7 The inefficient MERGE

```text
  5,000 new/changed rows
          |
          v
  +--------------------+       +----------------------------------+
  | Feed staging       | ----> | MERGE source                    |
  +--------------------+       | 1. Read all ledger rows         |
                                | 2. LAST_VALUE + ROW_NUMBER     |
                                | 3. Sort wide PAYLOAD column    |
                                +----------------+-----------------+
                                                 |
                              +------------------v------------------+
                              | Join on random TXN_KEY             |
                              | 0% pruning: every partition read |
                              +------------------+------------------+
                                                 |
                              +------------------v------------------+
                              | 20M-row ledger                    |
                              | large scan + local disk spill    |
                              +-------------------------------------+

  Small input does not imply a small query when the source subquery
  forces a full-table window pass before the MERGE can match keys.
```


**This is the centrepiece.** Three anti-patterns stacked together:

**A. A redundant full-table window pass inside the MERGE source.**
The subquery runs `LAST_VALUE()` and `ROW_NUMBER()` across all 20M rows just to
look up the prior payload for 5,000 keys. This is a copy/paste "get the latest
version" idiom, and it is the entire cause of the spill.

Note that `PRIOR_PAYLOAD` is genuinely consumed by the `UPDATE`. That detail
matters: if it were selected but unused, Snowflake's column pruning would
optimise the sort away and there would be no spill at all.

**B. The join is on `TXN_KEY`**, which has no cluster key and is randomly
distributed — so no partitions can be pruned.

**C. It runs on a fixed clock**, with no stream and no `WHEN` condition, so it
fires whether or not there is anything to process.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE PROCEDURE SP_MERGE_TRANSACTIONS(P_PIPELINE_NAME STRING)\nRETURNS STRING\nLANGUAGE SQL\nAS\n$$\nDECLARE\n    v_run_id   STRING DEFAULT UUID_STRING();\n    v_started  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()::TIMESTAMP_NTZ;\n    v_rows     NUMBER DEFAULT 0;\nBEGIN\n    CALL SP_GENERATE_FEED_BATCH(5000, :P_PIPELINE_NAME);\n\n    SELECT COUNT(*) INTO :v_rows\n    FROM TRANSACTION_FEED_STAGING\n    WHERE PIPELINE_NAME = :P_PIPELINE_NAME;\n\n    MERGE INTO TRANSACTION_LEDGER AS tgt\n    USING (\n        SELECT\n            s.TXN_KEY, s.ENTITY_ID, s.PRODUCT_ID, s.REGION,\n            s.QUANTITY, s.AMOUNT, s.TXN_STATUS, s.LAST_UPDATED,\n            COALESCE(d.PRIOR_PAYLOAD, s.PAYLOAD) AS PAYLOAD,\n            COALESCE(d.VERSION_NO, 0) + 1        AS NEXT_VERSION\n        FROM TRANSACTION_FEED_STAGING s\n        LEFT JOIN (\n            -- ANTI-PATTERN: full-table window pass on every single run\n            SELECT TXN_KEY, VERSION_NO,\n                   LAST_VALUE(PAYLOAD) OVER (PARTITION BY TXN_KEY\n                                             ORDER BY LAST_UPDATED\n                                             ROWS BETWEEN UNBOUNDED PRECEDING\n                                                      AND UNBOUNDED FOLLOWING) AS PRIOR_PAYLOAD,\n                   ROW_NUMBER() OVER (PARTITION BY TXN_KEY ORDER BY LAST_UPDATED DESC) AS RN\n            FROM TRANSACTION_LEDGER\n        ) d\n          ON d.TXN_KEY = s.TXN_KEY\n         AND d.RN = 1\n        WHERE s.PIPELINE_NAME = :P_PIPELINE_NAME\n    ) AS src\n      ON tgt.TXN_KEY = src.TXN_KEY     -- no pruning possible\n    WHEN MATCHED THEN UPDATE SET\n        tgt.QUANTITY     = src.QUANTITY,\n        tgt.AMOUNT       = src.AMOUNT,\n        tgt.TXN_STATUS   = src.TXN_STATUS,\n        tgt.VERSION_NO   = src.NEXT_VERSION,\n        tgt.LAST_UPDATED = src.LAST_UPDATED,\n        tgt.PAYLOAD      = src.PAYLOAD\n    WHEN NOT MATCHED THEN INSERT\n        (TXN_KEY, ENTITY_ID, PRODUCT_ID, REGION, QUANTITY,\n         AMOUNT, TXN_STATUS, VERSION_NO, LAST_UPDATED, PAYLOAD)\n    VALUES\n        (src.TXN_KEY, src.ENTITY_ID, src.PRODUCT_ID, src.REGION, src.QUANTITY,\n         src.AMOUNT, src.TXN_STATUS, 1, src.LAST_UPDATED, src.PAYLOAD);\n\n    INSERT INTO PIPELINE_RUN_LOG\n        (RUN_ID, PIPELINE_NAME, STARTED_AT, ENDED_AT, ROWS_IN_BATCH, STATUS, MESSAGE)\n    SELECT :v_run_id, :P_PIPELINE_NAME, :v_started,\n           CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, :v_rows, 'SUCCESS', NULL;\n\n    RETURN 'merged batch of ' || :v_rows || ' rows';\nEND;\n$$"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.8 Downstream reporting, and a task that fails

The reporting refresh full-scans the ledger every five minutes to rebuild a
summary, carrying a wide `MAX()` through a 2M-group aggregate — whether or not
anyone is reading the output.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE PROCEDURE SP_REFRESH_LEDGER_SUMMARY()\nRETURNS STRING\nLANGUAGE SQL\nAS\n$$\nBEGIN\n    CREATE OR REPLACE TABLE LEDGER_SUMMARY AS\n    SELECT\n        ENTITY_ID,\n        REGION,\n        COUNT(*)     AS TXN_COUNT,\n        SUM(AMOUNT)  AS TOTAL_AMOUNT,\n        SUM(CASE WHEN TXN_STATUS = 'FAILED' THEN AMOUNT ELSE 0 END) AS AMOUNT_AT_RISK,\n        MAX(PAYLOAD) AS LAST_PAYLOAD   -- drags the wide column through the aggregate\n    FROM TRANSACTION_LEDGER\n    GROUP BY ENTITY_ID, REGION\n    ORDER BY TOTAL_AMOUNT DESC;\n\n    RETURN 'summary refreshed';\nEND;\n$$"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


A second job fails intermittently — about 35% of runs.

Two things make this realistic. Because the task calls a **stored procedure**,
Snowflake cannot transparently retry it the way it would an ordinary statement,
so every failure surfaces. And `TASK_AUTO_RETRY_ATTEMPTS` is deliberately left
unset, so nothing self-heals.

The run-log insert autocommits before the exception is raised, so failed runs
are still recorded — giving two independent evidence trails to correlate.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE PROCEDURE SP_VARIANT_ENRICHMENT()\nRETURNS STRING\nLANGUAGE SQL\nAS\n$$\nDECLARE\n    SIM_FAILURE EXCEPTION (-20001, 'SQL execution internal error: processing aborted in enrichment step');\n    v_roll NUMBER;\nBEGIN\n    SELECT UNIFORM(1, 100, RANDOM()) INTO :v_roll;\n\n    INSERT INTO PIPELINE_RUN_LOG\n        (RUN_ID, PIPELINE_NAME, STARTED_AT, ENDED_AT, ROWS_IN_BATCH, STATUS, MESSAGE)\n    SELECT UUID_STRING(), 'T_VARIANT_ENRICHMENT',\n           CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, NULL, NULL,\n           CASE WHEN :v_roll <= 35 THEN 'FAILED' ELSE 'SUCCESS' END,\n           'roll=' || :v_roll;\n\n    IF (:v_roll <= 35) THEN\n        RAISE SIM_FAILURE;\n    END IF;\n\n    RETURN 'enrichment completed';\nEND;\n$$"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.9 Tasks

```text
  Scheduler
      |
      +--> T_INGEST_MERGE_XS  --> DEMO_INGEST_WH_XS  --> MERGE / spill
      |
      +--> T_INGEST_MERGE_S   --> DEMO_INGEST_WH_S   --> MERGE / spill
      |
      +--> T_REPORTING_REFRESH --> DEMO_REPORTING_WH --> full scan
      |
      +--> T_VARIANT_ENRICHMENT --> DEMO_REPORTING_WH --> intermittent failure

  All tasks are clock-driven. There is no stream predicate or WHEN condition.
  The same workload is intentionally run at two warehouse sizes so the
  demo can separate capacity improvement from cost/root-cause improvement.
```


**All four are created suspended.** Nothing recurring runs until Part 2.

Cadences are sized against measured runtimes so runs complete rather than being
skipped:

| Task | Warehouse | Measured runtime | Cadence |
|---|---|---|---|
| `T_INGEST_MERGE_XS` | X-Small | 137 s | 3 min |
| `T_INGEST_MERGE_S` | Small | 46 s | 1 min |
| `T_REPORTING_REFRESH` | X-Small | 15 s | 5 min |
| `T_VARIANT_ENRICHMENT` | X-Small | < 1 s | 2 min |

None of them has a stream predicate or a `WHEN` condition. That absence is
itself one of the findings worth surfacing.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TASK T_INGEST_MERGE_XS\n    WAREHOUSE = DEMO_INGEST_WH_XS\n    SCHEDULE = '3 MINUTE'\n    COMMENT = 'DEMO - fixed-cadence MERGE on X-Small, no change predicate'\nAS\n    CALL PIPELINE_COST_DEMO.INGESTION.SP_MERGE_TRANSACTIONS('INGEST_XS')"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TASK T_INGEST_MERGE_S\n    WAREHOUSE = DEMO_INGEST_WH_S\n    SCHEDULE = '1 MINUTE'\n    COMMENT = 'DEMO - identical work on a Small warehouse (the resized tier)'\nAS\n    CALL PIPELINE_COST_DEMO.INGESTION.SP_MERGE_TRANSACTIONS('INGEST_S')"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TASK T_REPORTING_REFRESH\n    WAREHOUSE = DEMO_REPORTING_WH\n    SCHEDULE = '5 MINUTE'\n    COMMENT = 'DEMO - full-table aggregate refresh on a fixed cadence'\nAS\n    CALL PIPELINE_COST_DEMO.INGESTION.SP_REFRESH_LEDGER_SUMMARY()"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["CREATE OR REPLACE TASK T_VARIANT_ENRICHMENT\n    WAREHOUSE = DEMO_REPORTING_WH\n    SCHEDULE = '2 MINUTE'\n    -- TASK_AUTO_RETRY_ATTEMPTS deliberately NOT set\n    COMMENT = 'DEMO - stored-procedure task that fails intermittently, no auto-retry'\nAS\n    CALL PIPELINE_COST_DEMO.INGESTION.SP_VARIANT_ENRICHMENT()"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 1.10 Seed the telemetry

Snowflake's `ACCOUNT_USAGE` views lag — query history by up to 45 minutes,
warehouse metering by up to 3 hours. This step runs the pipeline a few times
**right now** so that real-time telemetry has evidence immediately and you are
not waiting on view latency.

`EXECUTE TASK` triggers a suspended task once, on demand. Using it here rather
than calling the procedures directly guarantees each run lands on its own
warehouse, which is what makes the tier comparison in Part 3 valid.

These run asynchronously and take about 4 minutes to all finish.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['EXECUTE TASK T_INGEST_MERGE_XS',
    'EXECUTE TASK T_INGEST_MERGE_S',
    'EXECUTE TASK T_REPORTING_REFRESH',
    'EXECUTE TASK T_VARIANT_ENRICHMENT']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


Drop the build warehouse — the one-off generation work is done. Switch the
notebook session to a demo warehouse first; Snowflake does not allow a session
to drop the warehouse it is currently using.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['USE WAREHOUSE DEMO_REPORTING_WH',
    'DROP WAREHOUSE IF EXISTS DEMO_BUILD_WH']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


### Confirm the build

Expect ~20M rows and ~6.3 GB.

In [ ]:
%%sql -r verify_build
SELECT TABLE_NAME, ROW_COUNT, ROUND(BYTES/POWER(1024,3), 2) AS GB
FROM PIPELINE_COST_DEMO.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'INGESTION'
ORDER BY BYTES DESC NULLS LAST

---
# Part 2 — Start the pipeline

Running the cell below starts continuous consumption of roughly
**3.5 credits/hour**. The resource monitor caps it at 40 credits/day.

## How long to let it run

| Soaked for | What becomes demonstrable |
|---|---|
| Seed only | Query-level diagnosis (Prompt 2) |
| 15 min | Task failure analysis (Prompt 3) |
| 45-60 min | Account-level cost analysis (Prompt 1) — `ACCOUNT_USAGE` populated |
| 2-3 hrs | Credit trends and warehouse residency fully visible |

**Start this at least 60 minutes before you need to present.**

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER TASK T_INGEST_MERGE_XS    RESUME',
    'ALTER TASK T_INGEST_MERGE_S     RESUME',
    'ALTER TASK T_REPORTING_REFRESH  RESUME',
    'ALTER TASK T_VARIANT_ENRICHMENT RESUME']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['SHOW TASKS IN SCHEMA PIPELINE_COST_DEMO.INGESTION']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


---
# Part 3 — Confirm the problem is real

Run these before presenting. Every one should look bad. Keep the numbers to
hand so you can check whatever Cortex Code reports against ground truth.

## 3.1 The join key is completely unclustered

Look at `average_depth` against `total_partition_count`. When depth equals the
partition count, **every partition overlaps every other one** and pruning on
that column is impossible.

Measured: `total_partition_count` 415, `average_depth` 415.0.

In [ ]:
%%sql -r clustering_check
SELECT SYSTEM$CLUSTERING_INFORMATION('PIPELINE_COST_DEMO.INGESTION.TRANSACTION_LEDGER', '(TXN_KEY)')
    AS CLUSTERING_INFO

## 3.2 Scan volume and runtime, by tier

This is the single most important result in the notebook.

Measured: **X-Small 136.7 s / 12.59 GB** versus **Small 46.1 s / 12.59 GB**.

The resize cut runtime roughly threefold and changed bytes scanned by *nothing*.
It bought speed at double the hourly rate without touching the cause.

Copy a `QUERY_ID` from the output — you need it for 3.3.

In [ ]:
%%sql -r merge_runs
SELECT WAREHOUSE_NAME,
       WAREHOUSE_SIZE,
       ROUND(TOTAL_ELAPSED_TIME/1000, 1)     AS ELAPSED_SEC,
       ROUND(BYTES_SCANNED/POWER(1024,3), 2) AS GB_SCANNED,
       ROWS_PRODUCED                         AS ROWS_MERGED,
       QUERY_PARAMETERIZED_HASH,
       QUERY_ID
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
        END_TIME_RANGE_START => DATEADD(hour, -3, CURRENT_TIMESTAMP()),
        RESULT_LIMIT => 500))
WHERE QUERY_TYPE = 'MERGE'
  AND WAREHOUSE_NAME LIKE 'DEMO_INGEST%'
ORDER BY START_TIME DESC

## 3.3 Pruning and spill for one execution

**Paste a `QUERY_ID` from 3.2 into the cell below.**

Two things are worth knowing here, because both cost people real time:

- `INFORMATION_SCHEMA.QUERY_HISTORY` does **not** expose `PARTITIONS_SCANNED`
  or `BYTES_SPILLED_*`. In real time those come only from
  `GET_QUERY_OPERATOR_STATS`.
- In that output the spill metrics live under the **`spilling`** key, not `io`.
  Looking under `io` returns nulls and makes a badly spilling query look clean.

Measured on Small: `TableScan` 417 of 417 partitions, `WindowFunction`
**16.0 GB** local spill at 59% of runtime.
On X-Small: 415 of 415 partitions, **28.3 GB** spill at 63% of runtime.

In [ ]:
# Paste a QUERY_ID from the output of 3.2
MERGE_QUERY_ID = "<PASTE_QUERY_ID_HERE>"

In [ ]:
%%sql -r operator_stats
SELECT OPERATOR_TYPE,
       OPERATOR_STATISTICS:pruning:partitions_scanned::NUMBER AS PART_SCANNED,
       OPERATOR_STATISTICS:pruning:partitions_total::NUMBER   AS PART_TOTAL,
       ROUND(100 * OPERATOR_STATISTICS:pruning:partitions_scanned::NUMBER
                 / NULLIF(OPERATOR_STATISTICS:pruning:partitions_total::NUMBER, 0), 1)
           AS PCT_PARTITIONS_SCANNED,
       ROUND(OPERATOR_STATISTICS:spilling:bytes_spilled_local_storage::NUMBER
             / POWER(1024,3), 2) AS GB_SPILL_LOCAL,
       ROUND(OPERATOR_STATISTICS:spilling:bytes_spilled_remote_storage::NUMBER
             / POWER(1024,3), 2) AS GB_SPILL_REMOTE,
       OPERATOR_STATISTICS:output_rows::NUMBER AS OUTPUT_ROWS,
       ROUND(EXECUTION_TIME_BREAKDOWN:overall_percentage::FLOAT * 100, 1) AS PCT_OF_RUNTIME
FROM TABLE(GET_QUERY_OPERATOR_STATS('{{MERGE_QUERY_ID}}'))
ORDER BY PCT_OF_RUNTIME DESC NULLS LAST

## 3.4 Credits by warehouse

In [ ]:
%%sql -r credits_by_wh
SELECT WAREHOUSE_NAME,
       ROUND(SUM(CREDITS_USED_COMPUTE), 3)        AS COMPUTE_CREDITS,
       ROUND(SUM(CREDITS_USED_CLOUD_SERVICES), 3) AS CLOUD_SERVICES_CREDITS,
       ROUND(SUM(CREDITS_USED), 3)                AS TOTAL_CREDITS
FROM TABLE(INFORMATION_SCHEMA.WAREHOUSE_METERING_HISTORY(
        DATEADD(hour, -24, CURRENT_TIMESTAMP())))
WHERE WAREHOUSE_NAME LIKE 'DEMO_%'
GROUP BY WAREHOUSE_NAME
ORDER BY TOTAL_CREDITS DESC

## 3.5 Task outcomes

Run 15+ minutes after starting the pipeline. Expect `T_VARIANT_ENRICHMENT` to
have failed on roughly a third of its runs.

In [ ]:
%%sql -r task_outcomes
SELECT NAME,
       STATE,
       COUNT(*) AS RUNS,
       ROUND(AVG(DATEDIFF(millisecond, QUERY_START_TIME, COMPLETED_TIME))/1000, 1) AS AVG_SEC,
       MAX(LEFT(ERROR_MESSAGE, 90)) AS SAMPLE_ERROR
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD(hour, -3, CURRENT_TIMESTAMP()),
        RESULT_LIMIT => 500))
GROUP BY NAME, STATE
ORDER BY NAME, STATE

## 3.6 The pipeline's own run log

An application-level view, independent of Snowflake's task telemetry. Having
both matters in Part 4 — correlating them is how you tell a platform problem
apart from an application problem.

In [ ]:
%%sql -r run_log
SELECT PIPELINE_NAME,
       STATUS,
       COUNT(*) AS RUNS,
       ROUND(AVG(DATEDIFF(millisecond, STARTED_AT, ENDED_AT))/1000, 1) AS AVG_SEC,
       MIN(STARTED_AT) AS FIRST_RUN,
       MAX(STARTED_AT) AS LAST_RUN
FROM PIPELINE_COST_DEMO.INGESTION.PIPELINE_RUN_LOG
GROUP BY PIPELINE_NAME, STATUS
ORDER BY PIPELINE_NAME, STATUS

---
# Part 4 — The prompts

Everything above established that a real, measurable problem exists. Now hand it
to Cortex Code.

**Before you start:** set your context to this database. Running as
`ACCOUNTADMIN` with no database context makes "explore my account" surface every
unrelated database in the account, and the demo loses focus.

```sql
USE DATABASE PIPELINE_COST_DEMO;
USE SCHEMA INGESTION;
```

The three prompts follow a deliberate arc: **find it → diagnose it → deliver
something someone can act on.**

## Prompt 1 — Find the problem

```text
Read-only analysis of the PIPELINE_COST_DEMO database and the DEMO_* warehouses.

Analyze consumption over the last 24 hours. Identify which warehouses and which
recurring workloads are driving credits, and determine for each whether the cost
is caused by workload volume, warehouse sizing, inefficient data scanning, idle
warehouse residency, or failing and retrying jobs.

Note that INFORMATION_SCHEMA.QUERY_HISTORY does not expose partition or spill
metrics, and ACCOUNT_USAGE views lag by 45 minutes to 3 hours. Use whichever
source is appropriate and tell me which one you used.

Show the evidence: warehouse, size, credits, query counts, bytes scanned,
runtime, and task success and failure counts. Separate confirmed facts from
hypotheses, then give me a prioritized list of what to investigate next.

Do not change anything.
```

**What it should find:** the Small tier as the largest consumer, warehouses
resident almost continuously, MERGE queries reading gigabytes for tiny batches,
and a job failing about a third of the time.

**Worth saying out loud:** this is the difference between a dashboard telling you
the number went up and the platform telling you why. Note that it chose a
telemetry source and said which one — the lag between those views is exactly
where cost investigations usually go wrong.

## Prompt 2 — Diagnose it and quantify the fix

```text
Read-only performance investigation of the ingestion pipeline in
PIPELINE_COST_DEMO.INGESTION.

Find the dominant recurring MERGE pattern. For a representative execution, pull
the actual query profile with GET_QUERY_OPERATOR_STATS and report, per operator:
partitions scanned versus total, bytes scanned, bytes spilled to local and remote
storage, and share of total runtime. The spill statistics are under the
'spilling' key, not 'io'.

Then read the stored procedure definition behind it and explain which specific
part of the SQL causes each cost signature.

Compare the same workload on DEMO_INGEST_WH_XS against DEMO_INGEST_WH_S and tell
me whether the resize addressed the root cause or the symptom.

Finally, give me a prioritized fix list. For each fix, estimate the improvement
and state your confidence. Write the candidate SQL and a validation plan, but do
not execute DDL or modify anything yet.
```

**What it should find:** the `WindowFunction` operator consuming 59-63% of
runtime with 16-28 GB of spill, 100% of partitions scanned, and the resize
having changed runtime without changing scan volume.

### Then push on the answer

This is the most valuable exchange in the demo, because the obvious fix is wrong.

```text
On the clustering recommendation specifically: the incoming batch keys are
uniformly distributed across the entire key domain. Quantify the partition
pruning improvement you actually expect. If clustering will not help here, say
so, and tell me what would.
```

**Why this matters.** Removing the redundant window pass is dramatic — 46.1 s to
2.7 s, spill eliminated. But **pruning does not improve, and a cluster key on
`TXN_KEY` will not fix it**: 5,000 uniformly random keys against 420 partitions
puts roughly 12 matching keys in every partition, so every partition still has
to be read.

For random point-lookup MERGE patterns the real levers are the **search
optimization service**, reducing execution frequency, or changing the physical
design — not clustering. Watching it recalculate and correct itself is more
convincing than watching it be right first time.

## Prompt 3 — Turn it into a deliverable

```text
Investigate the task failures in PIPELINE_COST_DEMO.INGESTION over the last
3 hours.

Correlate TASK_HISTORY with the pipeline's own PIPELINE_RUN_LOG table and the
query history. Establish the failure rate, whether failures correlate with
warehouse contention, workload volume or neither, and whether the task is
configured to recover on its own.

Quantify the operational and credit impact of the failures separately from the
cost of the inefficient MERGE pattern. Do not merge the two findings.

Then produce, as a single markdown file:
1. an executive summary,
2. a technical root-cause analysis with the supporting telemetry,
3. prioritized remediation steps with rollback notes,
4. a monitoring and runbook checklist,
5. a short email suitable for sending to the pipeline owners.

Label confirmed evidence separately from assumptions. Keep the analysis
read-only.
```

**What it should find:** a ~35% failure rate uncorrelated with volume,
`TASK_AUTO_RETRY_ATTEMPTS` unset, and the idempotency caveat that has to be
settled before enabling retries.

**Worth saying out loud:** the output is not an answer, it is the deliverable —
analysis, runbook and the email. And it kept two separate problems separate
rather than blaming one on the other, which is the mistake that makes real
investigations take days.

---
# Part 5 — Prove the fix

```text
  BEFORE: clock-driven + full-table window + random-key join

  Scheduler --> 5K batch --> full ledger scan --> window sort --> spill
                                  |
                                  +--> 100% partitions scanned

  AFTER: event-driven + bounded source + measured validation

  Stream/WHEN --> changed rows --> direct MERGE --> no redundant window
                                      |
                                      +--> less scan, no spill, lower cadence

  Important nuance: removing the redundant window removes the spill, but
  random point-lookups may still require a physical-design decision.
```


Optional, and only worth doing with an engineering audience. It writes to the
demo database, so plan on running Part 5.3 afterwards.

## 5.1 Let Cortex Code implement it

```text
Implement your highest-confidence fix inside PIPELINE_COST_DEMO only. Create the
optimized procedure as a new object rather than replacing the existing one, run
both against the same feed batch, and show me a measured before-and-after of
runtime, bytes scanned, partitions scanned and spill.

Create every object inside PIPELINE_COST_DEMO so my teardown script catches it.
```

## 5.2 Or run the benchmark yourself

The cell below is the corrected MERGE with the redundant window pass removed.
Run it against the current feed batch and compare with 3.2.

Measured: **2.7 s versus 46.1 s, 6.30 GB versus 12.59 GB, zero spill versus
16 GB.** Note that partitions scanned stays at 420 of 420 — which is the point
made in Prompt 2.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['MERGE INTO PIPELINE_COST_DEMO.INGESTION.TRANSACTION_LEDGER AS tgt\nUSING PIPELINE_COST_DEMO.INGESTION.TRANSACTION_FEED_STAGING AS src\n  ON tgt.TXN_KEY = src.TXN_KEY\nWHEN MATCHED THEN UPDATE SET\n    tgt.QUANTITY     = src.QUANTITY,\n    tgt.AMOUNT       = src.AMOUNT,\n    tgt.TXN_STATUS   = src.TXN_STATUS,\n    tgt.VERSION_NO   = tgt.VERSION_NO + 1,\n    tgt.LAST_UPDATED = src.LAST_UPDATED\nWHEN NOT MATCHED THEN INSERT\n    (TXN_KEY, ENTITY_ID, PRODUCT_ID, REGION, QUANTITY,\n     AMOUNT, TXN_STATUS, VERSION_NO, LAST_UPDATED, PAYLOAD)\nVALUES\n    (src.TXN_KEY, src.ENTITY_ID, src.PRODUCT_ID, src.REGION, src.QUANTITY,\n     src.AMOUNT, src.TXN_STATUS, 1, src.LAST_UPDATED, src.PAYLOAD)']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


Re-run **3.2** and **3.3** now to see the before-and-after side by side. The
corrected MERGE appears as the most recent row.

## 5.3 Reset between sessions

If anything modified the pipeline, reset before presenting again.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_XS    SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_S     SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_REPORTING_REFRESH  SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_VARIANT_ENRICHMENT SUSPEND']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


Undo anything that may have been added to the ledger, then look for new objects.
Both `ALTER TABLE` statements error harmlessly if the property was never set.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER TABLE PIPELINE_COST_DEMO.INGESTION.TRANSACTION_LEDGER DROP CLUSTERING KEY']
results = []
for statement in statements:
    try:
        results.append(session.sql(statement).collect())
    except Exception as error:
        print(f'Ignored cleanup state: {error}')
results[-1] if results else None


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['SHOW PROCEDURES IN SCHEMA PIPELINE_COST_DEMO.INGESTION']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


Re-run cell **1.7** to restore the deliberately inefficient procedure, then
Part 2 to restart the pipeline. A full rebuild from Part 1 takes about
10 minutes and ~2 credits, which is a perfectly reasonable reset if anything
looks off.

---
# Part 6 — Stop and clean up

## 6.1 Stop — run this the moment the session ends

Suspends every task and aborts in-flight queries. Safe to run at any time,
including mid-demo. Leaves all objects in place so you can restart from Part 2.

At ~3.5 credits/hour, a pipeline left running over a weekend would hit the
40-credit cost guard — which then also stops it mid-demo the next time you
present. Stop it deliberately.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_XS    SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_S     SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_REPORTING_REFRESH  SUSPEND',
    'ALTER TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_VARIANT_ENRICHMENT SUSPEND']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_XS ABORT ALL QUERIES',
    'ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_S  ABORT ALL QUERIES',
    'ALTER WAREHOUSE IF EXISTS DEMO_REPORTING_WH ABORT ALL QUERIES']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


### Confirm everything has stopped

Every task should read `suspended`. The warehouses drain within five minutes
via `AUTO_SUSPEND`; the cell after this parks them immediately.

A warehouse that is already suspended returns
`Invalid state. Warehouse 'X' cannot be suspended.` That error is harmless and
simply means it was already parked.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_XS SUSPEND',
    'ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_S  SUSPEND',
    'ALTER WAREHOUSE IF EXISTS DEMO_REPORTING_WH SUSPEND']
results = []
for statement in statements:
    try:
        results.append(session.sql(statement).collect())
    except Exception as error:
        print(f'Ignored cleanup state: {error}')
results[-1] if results else None


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['SHOW TASKS IN SCHEMA PIPELINE_COST_DEMO.INGESTION']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


## 6.2 Total cost of the session

In [ ]:
%%sql -r session_cost
SELECT WAREHOUSE_NAME,
       ROUND(SUM(CREDITS_USED), 3) AS TOTAL_CREDITS
FROM TABLE(INFORMATION_SCHEMA.WAREHOUSE_METERING_HISTORY(
        DATEADD(day, -2, CURRENT_TIMESTAMP())))
WHERE WAREHOUSE_NAME LIKE 'DEMO_%'
GROUP BY WAREHOUSE_NAME
ORDER BY TOTAL_CREDITS DESC

## 6.3 Full teardown

Only when you are finished with the demo entirely. Removes every object this
notebook created.

Order matters: tasks first, then detach the resource monitor from the warehouses
(a monitor cannot be dropped while still assigned), then the warehouses, the
monitor, and finally the database.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['DROP TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_XS',
    'DROP TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_INGEST_MERGE_S',
    'DROP TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_REPORTING_REFRESH',
    'DROP TASK IF EXISTS PIPELINE_COST_DEMO.INGESTION.T_VARIANT_ENRICHMENT']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_XS UNSET RESOURCE_MONITOR',
    'ALTER WAREHOUSE IF EXISTS DEMO_INGEST_WH_S  UNSET RESOURCE_MONITOR',
    'ALTER WAREHOUSE IF EXISTS DEMO_REPORTING_WH UNSET RESOURCE_MONITOR']
results = []
for statement in statements:
    try:
        results.append(session.sql(statement).collect())
    except Exception as error:
        print(f'Ignored cleanup state: {error}')
results[-1] if results else None


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ['DROP WAREHOUSE IF EXISTS DEMO_INGEST_WH_XS',
    'DROP WAREHOUSE IF EXISTS DEMO_INGEST_WH_S',
    'DROP WAREHOUSE IF EXISTS DEMO_REPORTING_WH',
    'DROP WAREHOUSE IF EXISTS DEMO_BUILD_WH',
    'DROP RESOURCE MONITOR IF EXISTS DEMO_COST_GUARD',
    'DROP DATABASE IF EXISTS PIPELINE_COST_DEMO']
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


### Confirm the teardown

All three should return no rows.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
statements = ["SHOW WAREHOUSES LIKE 'DEMO_%'"]
results = []
for statement in statements:
    results.append(session.sql(statement).collect())
results[-1]


### Checking for strays

If Cortex Code created objects **outside** `PIPELINE_COST_DEMO` during the
session — a warehouse, a resource monitor, a budget — the teardown above will
not have caught them. Worth a look:

```sql
SHOW WAREHOUSES;
SHOW RESOURCE MONITORS;
SHOW TASKS IN ACCOUNT;
```

Telling Cortex Code up front to create everything inside `PIPELINE_COST_DEMO`
keeps the blast radius inside the teardown.

---

## Reference — measured ground truth

Every figure below is a reference baseline from the equivalent validated workload.
Run Part 3 in your account to measure the exact values for this notebook.


| Signal | Value |
|---|---|
| Ledger table | 20M rows, 6.3 GB, ~415 micro-partitions |
| Clustering depth on the join key | 415 of 415 partitions |
| Input per pipeline run | 5,000 rows |
| Data read per pipeline run | **12.59 GB** |
| Partitions scanned per run | **417 of 417 (100%)** |
| MERGE runtime, Small | 46.1 s |
| MERGE runtime, X-Small | 136.7 s |
| Spill, Small | **16.0 GB** local, 59% of runtime |
| Spill, X-Small | **28.3 GB** local, 63% of runtime |
| Reporting refresh | 14.5 s, 6.3 GB, 2.0M rows, every 5 min |
| Variant task failure rate | ~35%, no auto-retry |
| Run rate, all tasks active | ~3.5 credits/hour |

### The corrected MERGE, for comparison

| | Inefficient | Corrected | Change |
|---|---|---|---|
| Runtime (Small) | 46.1 s | **2.7 s** | 17x faster |
| GB scanned | 12.59 | **6.30** | halved |
| Local spill | 16.0 GB | **0** | eliminated |
| Partitions scanned | 417 / 417 | **420 / 420** | **unchanged** |